# 05 · Tree Models: Decision Tree, Random Forest, Gradient Boosting

**Goal:** fit the non-linear models (lessons 14 and 15), which can follow the curved age–price relation and
combinations of features (a diesel pickup is worth more than "diesel" plus "pickup" separately), and compare them
with the linear models from notebook 04.

All models use the tree preprocessor (median-impute numbers, one-hot categories, no scaling) and `log(price)` as
the target; predictions are turned back into dollars before scoring.

**Tuning strategy.** The training set has 166,115 rows × 439 columns. Random Forest and Gradient Boosting take
minutes per fit at that size, so a full grid search would take hours. Following lesson 13, hyper-parameters are
chosen with `GridSearchCV` (3-fold) on a **random sample of 40,000 training rows**, then the best setting is refit
**once on the full training set** and scored once on the test set. The Decision Tree is fast enough to tune on the
full training set.

⏱️ *Runtime: about 30–40 minutes on a laptop, mostly the final Random Forest and Gradient Boosting fits.*

In [ ]:
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

sys.path.append('../src')
from prep import (load_data, split, add_model_top, tree_preprocessor, evaluate,
                  save_result, save_params, RANDOM_STATE)

sns.set_theme(style='whitegrid')
df = load_data()
X_train, X_test, y_train, y_test = split(df)
X_train, X_test = add_model_top(X_train, X_test)
y_train_log = np.log(y_train)

sample = X_train.sample(40_000, random_state=RANDOM_STATE).index
X_tune, y_tune_log = X_train.loc[sample], y_train_log.loc[sample]
print('train', X_train.shape, '| test', X_test.shape, '| tuning sample', X_tune.shape)

test_preds = pd.DataFrame({'price': y_test})
train_vs_test = []

def score(name, model):
    # Score on test (once) and on train, to show overfitting; store the predictions
    pred_test = np.exp(model.predict(X_test))
    pred_train = np.exp(model.predict(X_train))
    test_preds[name] = pred_test.round(0)
    train_vs_test.append({'model': name,
                          'train_mae': mean_absolute_error(y_train, pred_train),
                          'test_mae': mean_absolute_error(y_test, pred_test)})
    return save_result(evaluate(name, y_test, pred_test))

## 1. Decision Tree (CART regression tree, lesson 14)

### A shallow tree to see how it splits
A depth-3 tree is too simple to predict well, but it shows which questions the model asks first.

In [ ]:
small = make_pipeline(tree_preprocessor(), DecisionTreeRegressor(max_depth=3, random_state=RANDOM_STATE))
small.fit(X_train, y_train_log)

plt.figure(figsize=(22, 8))
plot_tree(small[-1], feature_names=small[0].get_feature_names_out(), filled=True, rounded=True,
          impurity=False, precision=2, fontsize=10)
plt.title('Depth-3 regression tree (values are log(price); exp(9.2) ≈ $9,900, exp(10.0) ≈ $22,000)')
plt.show()

SHALLOW_MD

### Tuning the full tree

In [ ]:
t = time.time()
dt = GridSearchCV(make_pipeline(tree_preprocessor(), DecisionTreeRegressor(random_state=RANDOM_STATE)),
                  {'decisiontreeregressor__max_depth': [10, 15, 20, 30],
                   'decisiontreeregressor__min_samples_leaf': [5, 20, 50]},
                  cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
dt.fit(X_train, y_train_log)
print(f'{time.time() - t:.0f} s | best:', dt.best_params_)

res = pd.DataFrame(dt.cv_results_)
res.pivot(index='param_decisiontreeregressor__max_depth',
          columns='param_decisiontreeregressor__min_samples_leaf', values='mean_test_score').round(4)

In [ ]:
save_params('Decision Tree', dt.best_params_)
score('Decision Tree', dt.best_estimator_)

## 2. Random Forest (lesson 15)

Many deep trees, each trained on a bootstrap sample and a random subset of columns at every split; their
average is much more stable than one tree.

In [ ]:
t = time.time()
rf_search = GridSearchCV(
    make_pipeline(tree_preprocessor(), RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
    {'randomforestregressor__max_features': [0.3, 0.6],
     'randomforestregressor__min_samples_leaf': [1, 3]},
    cv=3, scoring='neg_mean_absolute_error')
rf_search.fit(X_tune, y_tune_log)
print(f'{time.time() - t:.0f} s | best on the sample:', rf_search.best_params_)
pd.DataFrame(rf_search.cv_results_)[['param_randomforestregressor__max_features',
                                     'param_randomforestregressor__min_samples_leaf', 'mean_test_score']].round(4)

In [ ]:
t = time.time()
rf = rf_search.best_estimator_.fit(X_train, y_train_log)     # refit the best setting on all training rows
print(f'Final fit on {len(X_train):,} rows: {time.time() - t:.0f} s')
save_params('Random Forest', rf_search.best_params_)
score('Random Forest', rf)

## 3. Gradient Boosting (lesson 15)

Trees are added one at a time, each one correcting the errors of the trees before it. Shallow trees, a small
learning rate and `subsample=0.8` (each tree sees 80% of the rows) keep it from overfitting.

In [ ]:
t = time.time()
gb_search = GridSearchCV(
    make_pipeline(tree_preprocessor(),
                  GradientBoostingRegressor(learning_rate=0.1, subsample=0.8, random_state=RANDOM_STATE)),
    {'gradientboostingregressor__n_estimators': [300, 600],
     'gradientboostingregressor__max_depth': [4, 6]},
    cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
gb_search.fit(X_tune, y_tune_log)
print(f'{time.time() - t:.0f} s | best on the sample:', gb_search.best_params_)
pd.DataFrame(gb_search.cv_results_)[['param_gradientboostingregressor__n_estimators',
                                     'param_gradientboostingregressor__max_depth', 'mean_test_score']].round(4)

In [ ]:
t = time.time()
gb = gb_search.best_estimator_.fit(X_train, y_train_log)
print(f'Final fit on {len(X_train):,} rows: {time.time() - t:.0f} s')
save_params('Gradient Boosting', gb_search.best_params_)
score('Gradient Boosting', gb)

## 4. Overfitting check: training error vs test error

In [ ]:
tvt = pd.DataFrame(train_vs_test).set_index('model')
tvt['gap'] = tvt['test_mae'] - tvt['train_mae']
tvt.round(0)

GAP_MD

## 5. All models so far (test set)

In [ ]:
test_preds.to_csv('../results/test_predictions.csv', index=False)
results = pd.read_csv('../results/metrics.csv').sort_values('mae')
results.round(3)

In [ ]:
plot = results.sort_values('mae', ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(plot['model'], plot['mae'], color='steelblue')
axes[0].bar_label(axes[0].containers[0], fmt='$%.0f', padding=3)
axes[0].set_title('Test MAE in dollars (lower is better)')
axes[0].set_xlim(0, plot['mae'].max() * 1.15)
axes[1].barh(plot['model'], plot['r2'].clip(lower=0), color='steelblue')
axes[1].bar_label(axes[1].containers[0], fmt='%.3f', padding=3)
axes[1].set_title('Test R² (higher is better)')
axes[1].set_xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Error by price band for the best model: where is the model weakest?
best = results.iloc[0]['model']
bands = pd.cut(test_preds['price'], [0, 5000, 10000, 20000, 35000, 150000],
               labels=['< $5k', '$5-10k', '$10-20k', '$20-35k', '> $35k'])
err = (test_preds[best] - test_preds['price']).abs()
by_band = pd.DataFrame({'cars': bands.value_counts(sort=False),
                        'mae_$': err.groupby(bands, observed=True).mean().round(0),
                        'mae_%_of_price': (err / test_preds['price'] * 100).groupby(bands, observed=True).mean().round(1)})
print('Best model:', best)
by_band

SUMMARY_MD